# Collecte GTFS : arrêts, lignes et horaires des transports en commun

Charge le catalogue des GTFS publiés, l'agrégat national de tous les arrêts de France, puis les GTFS complets (lignes, courses, horaires) des réseaux de la zone d'étude.

**Prérequis** : un Lakehouse attaché comme Lakehouse par défaut du notebook.
Les fichiers bruts sont écrits dans `Files/raw/`, les tables Delta dans `Tables/`.

*Notebook généré depuis `src/ingestion/` par `fabric/build_notebooks.py` : modifier les scripts, pas ce notebook.*

In [ ]:
# Zone d'étude : Bordeaux Métropole (TBM) + réseaux de Gironde
ZONE = ["Bordeaux Métropole", "Gironde", "Aéroport de Bordeaux"]  # ["*"] = toute la France
EXCLURE_AGREGATS = True  # écarte les agrégats régionaux qui dupliquent les réseaux locaux
AVEC_STOP_TIMES = True  # horaires détaillés (volumineux)
FORCE = False


## Outils communs

In [ ]:
"""Outils communs à la collecte : chemins, téléchargements HTTP, écriture des tables.

Les mêmes scripts tournent en local (sortie dans data/raw/) et dans un notebook
Microsoft Fabric (sortie dans le Lakehouse attaché : /lakehouse/default/Files/raw/).
"""
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

FABRIC_LAKEHOUSE = Path("/lakehouse/default/Files/raw")
LOCAL_RAW = Path(__file__).resolve().parents[2] / "data" / "raw" if "__file__" in globals() else Path("data/raw")

USER_AGENT = "hackathon-mobility-citiz/1.0 (collecte open data)"
PAN_API = "https://transport.data.gouv.fr/api/datasets"


def raw_dir() -> Path:
    """Dossier racine des données brutes : Lakehouse Fabric si présent, sinon data/raw/."""
    base = Path(os.environ["RAW_DIR"]) if "RAW_DIR" in os.environ else (
        FABRIC_LAKEHOUSE if FABRIC_LAKEHOUSE.parent.exists() else LOCAL_RAW
    )
    base.mkdir(parents=True, exist_ok=True)
    return base


def tables_dir() -> Path:
    """Tables Parquet prêtes à être chargées en tables Delta dans Fabric."""
    d = raw_dir() / "tables"
    d.mkdir(parents=True, exist_ok=True)
    return d


def now_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def http_get(url: str, timeout: int = 60, retries: int = 3, **kwargs) -> requests.Response:
    headers = {"User-Agent": USER_AGENT, **kwargs.pop("headers", {})}
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout, **kwargs)
            r.raise_for_status()
            return r
        except requests.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


def get_json(url: str, timeout: int = 30):
    return http_get(url, timeout=timeout).json()


def download(url: str, dest: Path, force: bool = False, version: str | None = None) -> Path:
    """Télécharge un fichier en streaming. Ne retélécharge pas s'il existe déjà, sauf si
    `version` (date de mise à jour, empreinte…) diffère de celle du fichier en cache."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    fichier_version = dest.with_name(dest.name + ".version")
    en_cache = fichier_version.read_text() if fichier_version.exists() else None
    if dest.exists() and dest.stat().st_size > 0 and not force and (version is None or version == en_cache):
        print(f"  déjà présent : {dest.name}")
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")
    with http_get(url, timeout=300, stream=True) as r, open(tmp, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    tmp.rename(dest)
    if version is not None:
        fichier_version.write_text(version)
    elif fichier_version.exists():
        fichier_version.unlink()
    print(f"  téléchargé : {dest.name} ({dest.stat().st_size / 1e6:.1f} Mo)")
    return dest


def save_json(obj, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(json.dumps(obj, ensure_ascii=False))


def _manquant(v) -> bool:
    """Valeur manquante (None, NaN, pd.NA), sans échouer sur les listes ou dictionnaires des flux JSON."""
    return v is None or v is pd.NA or (isinstance(v, float) and v != v)


def _types_homogenes(df: pd.DataFrame) -> pd.DataFrame:
    """Les flux open data mélangent parfois nombres et textes dans une même colonne :
    on convertit ces colonnes en texte pour que Parquet / Delta les acceptent.
    Les colonnes booléennes avec des valeurs manquantes restent booléennes (type nullable)."""
    for col in df.columns[df.dtypes == object]:
        valeurs = df[col].dropna()
        types = {type(v) for v in valeurs}
        if not types:  # colonne entièrement vide : type "null" refusé par Delta, on la type en texte
            df[col] = df[col].astype("string")
        elif types & {bool, np.bool_} and types <= {bool, np.bool_, str} and valeurs.map(
                lambda v: str(v).lower() in ("true", "false")).all():
            # booléens avec des trous (ou déjà écrits en texte "True"/"False" dans l'historique)
            df[col] = df[col].map(lambda v: None if _manquant(v) else str(v).lower() == "true").astype("boolean")
        elif all(issubclass(t, (int, float, np.integer, np.floating)) and not issubclass(t, (bool, np.bool_))
                 for t in types):  # entiers et décimaux mélangés : nombre décimal
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
        elif len(types) > 1 or types - {str}:
            df[col] = df[col].map(lambda v: None if _manquant(v) else str(v))
    return df


# Tables écrites pendant l'exécution en cours : le notebook Fabric ne publie que celles-ci.
TABLES_ECRITES: list[str] = []


def write_table(df: pd.DataFrame, name: str, historique: bool = False) -> Path:
    """Écrit une table Parquet dans <raw>/tables/<name>.parquet.

    historique=True : la table ne contient que la photo de cette exécution ; elle est aussi
    archivée dans <raw>/historique/<name>/<date_collecte>.parquet, sans relire les photos
    précédentes (le notebook Fabric l'ajoute à la table Delta au lieu de la réécrire)."""
    path = tables_dir() / f"{name}.parquet"
    df = _types_homogenes(df.copy())  # ne modifie pas le DataFrame de l'appelant
    df.to_parquet(path, index=False)
    if historique:
        archive = raw_dir() / "historique" / name / f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.parquet"
        archive.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(archive, index=False)
    if name not in TABLES_ECRITES:
        TABLES_ECRITES.append(name)
    print(f"  table {name} : {len(df):,} lignes -> {path}")
    return path


def pan_catalog() -> list:
    """Catalogue complet du Point d'Accès National transport.data.gouv.fr."""
    return get_json(PAN_API, timeout=120)


## Collecte

In [ ]:
"""Collecte GTFS : arrêts, lignes et horaires des transports en commun.

Sources :
  - Catalogue du Point d'Accès National : https://transport.data.gouv.fr/datasets?type=public-transit
  - Agrégat national des arrêts : https://www.data.gouv.fr/datasets/arrets-de-transport-en-france

Trois niveaux de collecte :
  1. le catalogue de tous les GTFS publiés (léger) ;
  2. tous les arrêts de France en un seul fichier (environ 440 Mo, non dédoublonné) ;
  3. les GTFS complets (lignes, courses, horaires) des réseaux de la zone d'étude.

Sorties :
  <raw>/gtfs/arrets_france.csv
  <raw>/gtfs/zips/<jeu>_<ressource>.zip
  <raw>/tables/gtfs_catalogue.parquet
  <raw>/tables/gtfs_arrets_france.parquet
  <raw>/tables/gtfs_ressources.parquet  (statut de lecture de chaque GTFS de la zone)
  <raw>/tables/gtfs_<fichier>.parquet   (agency, routes, stops, trips, calendar, calendar_dates, stop_times)
"""
import zipfile

import pandas as pd


ARRETS_FRANCE_DATASET = "https://www.data.gouv.fr/api/1/datasets/arrets-de-transport-en-france/"
# Zone d'étude retenue : Bordeaux Métropole (TBM) + réseaux de Gironde pour le périurbain
ZONE_BORDEAUX = ["Bordeaux Métropole", "Gironde", "Aéroport de Bordeaux"]
FICHIERS_GTFS = ["agency", "routes", "stops", "trips", "calendar", "calendar_dates", "stop_times"]


def catalogue() -> pd.DataFrame:
    rows = []
    for ds in pan_catalog():
        if ds.get("type") != "public-transit":
            continue
        zones = ds.get("covered_area") or []
        for r in ds.get("resources", []):
            if r.get("format") != "GTFS":
                continue
            meta = r.get("metadata") or {}
            rows.append({
                "jeu_id": ds["id"],
                "jeu_slug": ds["slug"],
                "jeu_titre": ds["title"],
                "producteur": (ds.get("publisher") or {}).get("name"),
                "licence": ds.get("licence"),
                "zones_couvertes": " | ".join(z.get("nom", "") for z in zones),
                "zones_insee": " | ".join(str(z.get("insee", "")) for z in zones),
                "types_zones": " | ".join(z.get("type", "") for z in zones),
                "ressource_id": r.get("id"),
                "ressource_titre": r.get("title"),
                "modes": ",".join(meta.get("modes") or []),
                "reseaux": ",".join(meta.get("networks") or []),
                "date_fin_validite": meta.get("end_date"),
                "disponible": r.get("is_available"),
                "mise_a_jour": r.get("updated"),
                "url": r.get("url"),
                "page_pan": ds.get("page_url"),
            })
    df = pd.DataFrame(rows)
    df["date_collecte"] = now_utc()
    write_table(df, "gtfs_catalogue")
    return df


def arrets_france(force: bool = False, cat: pd.DataFrame | None = None) -> pd.DataFrame:
    """Agrégat national de tous les arrêts GTFS (dernière version publiée sur data.gouv.fr).
    cat : catalogue GTFS, pour indiquer la licence du réseau d'origine de chaque arrêt."""
    meta = get_json(ARRETS_FRANCE_DATASET)
    ressource = next(r for r in meta["resources"] if r["format"] == "csv")
    print(f"  agrégat national : {ressource['title']}")
    # empreinte publiée par data.gouv.fr : un nouvel export remplace le fichier en cache
    version = (ressource.get("checksum") or {}).get("value") or ressource.get("last_modified")
    chemin = download(ressource["url"], raw_dir() / "gtfs" / "arrets_france.csv", force=force, version=version)
    df = pd.read_csv(chemin, dtype=str, low_memory=False)
    for col in ("stop_lat", "stop_lon"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["fichier_source"] = ressource["title"]
    if cat is not None:  # licence propre à chaque réseau (Licence Ouverte, ODbL…), absente si le GTFS a été retiré
        licences = dict(zip(cat["ressource_id"].astype(str), cat["licence"]))
        df["licence_reseau"] = df["resource_id"].map(licences)
    df["date_collecte"] = now_utc()
    write_table(df, "gtfs_arrets_france")
    return df


def selectionner(cat: pd.DataFrame, zone: list[str], exclure_agregats: bool = True) -> pd.DataFrame:
    """Filtre le catalogue sur des mots-clés (nom de zone, de réseau, titre ou code INSEE).
    zone = ["*"] sélectionne tous les GTFS de France.
    exclure_agregats : écarte les agrégats régionaux, qui dupliquent les réseaux locaux."""
    cat = cat[cat["disponible"].astype("boolean").fillna(False).astype(bool)]
    if exclure_agregats:
        cat = cat[~cat["jeu_titre"].str.lower().str.startswith("agrégat")]
    if zone == ["*"]:
        return cat
    texte = (cat["jeu_titre"] + " " + cat["zones_couvertes"] + " " + cat["zones_insee"] + " "
             + cat["reseaux"] + " " + cat["producteur"].fillna("")).str.lower()
    masque = pd.Series(False, index=cat.index)
    for mot in zone:
        masque |= texte.str.contains(mot.lower(), regex=False)
    return cat[masque]


def gtfs_complets(cat: pd.DataFrame, zone: list[str], avec_stop_times: bool = True, force: bool = False,
                  exclure_agregats: bool = True) -> None:
    choix = selectionner(cat, zone, exclure_agregats)
    print(f"  {len(choix)} GTFS sélectionnés pour la zone {zone}")
    fichiers = FICHIERS_GTFS if avec_stop_times else [f for f in FICHIERS_GTFS if f != "stop_times"]
    morceaux = {f: [] for f in fichiers}
    statuts = []
    for r in choix.to_dict("records"):
        zip_path = raw_dir() / "gtfs" / "zips" / f"{r['jeu_slug']}_{r['ressource_id']}.zip"
        lus = {}
        try:
            download(r["url"], zip_path, force=force, version=r.get("mise_a_jour"))
            with zipfile.ZipFile(zip_path) as z:
                noms = {n.split("/")[-1]: n for n in z.namelist()}
                for f in fichiers:
                    if f"{f}.txt" in noms:
                        with z.open(noms[f"{f}.txt"]) as fh:
                            df = pd.read_csv(fh, dtype=str, encoding="utf-8-sig")
                        df.insert(0, "ressource_id", str(r["ressource_id"]))
                        df.insert(1, "jeu_titre", r["jeu_titre"])
                        lus[f] = df
            statut = "ok"
        except Exception as e:
            print(f"  ! {r['jeu_titre']} ({r['ressource_id']}) ignoré : {e}")
            statut, lus = f"erreur: {e}", {}
        # une ressource n'est publiée que si tous ses fichiers ont été lus : pas d'arrêts sans courses
        for f, df in lus.items():
            morceaux[f].append(df)
        statuts.append({"ressource_id": str(r["ressource_id"]), "jeu_titre": r["jeu_titre"], "url": r["url"],
                        "licence": r.get("licence"), "mise_a_jour": r.get("mise_a_jour"), "statut": statut, "fichiers": ",".join(lus),
                        "nb_arrets": len(lus["stops"]) if "stops" in lus else None})
    horodatage = now_utc()
    colonnes = ["ressource_id", "jeu_titre", "url", "licence", "mise_a_jour", "statut", "fichiers", "nb_arrets"]
    write_table(pd.DataFrame(statuts, columns=colonnes).assign(date_collecte=horodatage), "gtfs_ressources")
    for f, dfs in morceaux.items():
        if dfs:
            df = pd.concat(dfs, ignore_index=True).assign(date_collecte=horodatage)
            for col in ("stop_lat", "stop_lon"):
                if col in df:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
            write_table(df, f"gtfs_{f}")


def collecter(zone: list[str], avec_stop_times: bool = True, force: bool = False,
              exclure_agregats: bool = True) -> None:
    print("GTFS - transports en commun")
    cat = catalogue()
    arrets_france(force=force, cat=cat)
    gtfs_complets(cat, zone, avec_stop_times=avec_stop_times, force=force, exclure_agregats=exclure_agregats)


In [ ]:
TABLES_ECRITES.clear()  # seules les tables de cette collecte seront publiées
collecter(zone=ZONE, avec_stop_times=AVEC_STOP_TIMES, force=FORCE, exclure_agregats=EXCLURE_AGREGATS)


## Chargement en tables Delta

In [ ]:
TABLES = ['gtfs_catalogue', 'gtfs_arrets_france', 'gtfs_ressources', 'gtfs_agency', 'gtfs_routes', 'gtfs_stops', 'gtfs_trips', 'gtfs_calendar', 'gtfs_calendar_dates', 'gtfs_stop_times']
HISTORIQUES = []

# Chargement en tables Delta du Lakehouse par défaut, à partir des Parquet écrits par CETTE exécution :
# - table produite : remplacée ; pour un historique, la photo est ajoutée à la suite (sans relire l'existant) ;
# - table non produite (ex. ZONE modifiée, AVEC_STOP_TIMES=False) : supprimée, pour ne jamais publier
#   d'anciennes données ; un historique sans nouvelle photo reste inchangé.
# L'écriture passe par l'adresse OneLake (abfss) : le dossier monté /lakehouse/default/Tables
# interrompt les envois de gros fichiers.
import shutil
from pathlib import Path
import pyarrow.parquet as pq
from deltalake import DeltaTable, Schema, write_deltalake

if not TABLES_ECRITES:
    raise RuntimeError("Aucune table produite dans cette session : exécuter d'abord la cellule de collecte.")

try:
    import notebookutils
    ctx = notebookutils.runtime.context
    TABLES_DELTA = (f"abfss://{ctx['defaultLakehouseWorkspaceId']}@onelake.dfs.fabric.microsoft.com/"
                    f"{ctx['defaultLakehouseId']}/Tables" + ("/dbo" if Path("/lakehouse/default/Tables/dbo").exists() else ""))
    OPTIONS = {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}

    def supprimer(chemin):
        if notebookutils.fs.exists(chemin):
            notebookutils.fs.rm(chemin, True)
except ImportError:  # hors Fabric : tables Delta locales, pour tester
    TABLES_DELTA, OPTIONS = str(raw_dir() / "delta"), None

    def supprimer(chemin):
        shutil.rmtree(chemin, ignore_errors=True)

for nom in TABLES + HISTORIQUES:
    cible, source = f"{TABLES_DELTA}/{nom}", raw_dir() / "tables" / f"{nom}.parquet"
    if nom not in TABLES_ECRITES:
        if nom in HISTORIQUES:
            print(f"{nom} : pas de nouvelle photo, historique inchangé")
        else:
            supprimer(cible)
            source.unlink(missing_ok=True)
            print(f"{nom} : non produite par cette exécution, table supprimée")
        continue
    donnees = pq.read_table(source)
    if nom not in HISTORIQUES:
        write_deltalake(cible, donnees, mode="overwrite", schema_mode="overwrite", storage_options=OPTIONS)
        print(f"{nom} : table Delta remplacée ({donnees.num_rows:,} lignes)")
        continue
    # photo déjà publiée (cellule relancée après un échec partiel) : pas de doublon
    photo = donnees.column("date_collecte")[0].as_py() if "date_collecte" in donnees.column_names and donnees.num_rows else None
    existe = DeltaTable.is_deltatable(cible, storage_options=OPTIONS)
    if existe and photo and len(DeltaTable(cible, storage_options=OPTIONS).to_pandas(
            columns=["date_collecte"], filters=[("date_collecte", "=", photo)])):
        print(f"{nom} : photo du {photo} déjà présente, rien à ajouter")
        continue
    # une colonne dont le type change (ex. booléens écrits en texte par une version précédente)
    # serait convertie silencieusement : on réécrit alors l'historique une fois, avec les types imposés
    if existe:
        existant = {f.name: str(f.type) for f in DeltaTable(cible, storage_options=OPTIONS).schema().fields}
        nouveau = {f.name: str(f.type) for f in (Schema.from_arrow(donnees.schema) if hasattr(Schema, "from_arrow")
                                                 else Schema.from_pyarrow(donnees.schema)).fields}
        conflits = [c for c in nouveau if c in existant and existant[c] != nouveau[c]]
    else:
        conflits = []
    if conflits:
        print(f"{nom} : types modifiés ({', '.join(conflits)}), réécriture unique de l'historique")
        ancien = DeltaTable(cible, storage_options=OPTIONS).to_pandas()
        tout = pd.concat([ancien.astype({c: object for c in conflits}), donnees.to_pandas()], ignore_index=True)
        if "_typer" in globals():  # types déclarés par le module de collecte (TYPES_HISTORIQUES du GBFS)
            tout = _typer(tout)
        write_deltalake(cible, _types_homogenes(tout), mode="overwrite", schema_mode="overwrite", storage_options=OPTIONS)
    else:
        write_deltalake(cible, donnees, mode="append", schema_mode="merge", storage_options=OPTIONS)
    print(f"{nom} : {donnees.num_rows:,} lignes ajoutées à l'historique")
